In [9]:
!pip install torch torchvision streamlit

In [14]:

!rm -rf dataset
!unzip -q -o "Garbage classification.zip"

In [15]:
!pip install split-folders

In [16]:
import splitfolders
splitfolders.ratio("Garbage classification", output="dataset", seed=42, ratio=(0.8, 0.2))

Copying files: 2527 files [00:00, 5847.42 files/s]


In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}
image_datasets = {x: datasets.ImageFolder(f'dataset/{x}', data_transforms[x]) for x in ['train', 'val']}
dataloaders = {x: torch.utils.data.DataLoader(image_datasets[x], batch_size=32, shuffle=True) for x in ['train', 'val']}
class_names = image_datasets['train'].classes
model = models.resnet50(weights = models.ResNet50_Weights.DEFAULT)
for param in model.parameters():
    param.requires_grad = False
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(class_names))
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Bắt đầu huấn luyện (chỉ train lớp cuối)...")
num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in dataloaders['train']:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f'Epoch {epoch+1}/{num_epochs} - Loss: {running_loss/len(dataloaders["train"]):.4f}')
torch.save(model.state_dict(), 'resnet50_eco_sorter.pth')

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 60.8MB/s]


Bắt đầu huấn luyện (chỉ train lớp cuối)...
Epoch 1/5 - Loss: 1.1401
Epoch 2/5 - Loss: 0.7010
Epoch 3/5 - Loss: 0.5489
Epoch 4/5 - Loss: 0.4616
Epoch 5/5 - Loss: 0.4137


In [19]:
%%writefile app.py
import streamlit as st
import torch
from torchvision import models, transforms
from PIL import Image

class_names = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

st.set_page_config(page_title="Phân loại rác AI", page_icon="♻️")
st.title("♻️ Trợ lý Phân loại Rác thải AI")
st.write("Hãy tải lên một bức ảnh rác, AI của chúng ta sẽ giúp bạn phân loại nó!")

@st.cache_resource
def load_model():
    model = models.resnet50(weights=None)
    num_ftrs = model.fc.in_features
    model.fc = torch.nn.Linear(num_ftrs, 6)
    model.load_state_dict(torch.load('resnet50_eco_sorter.pth', map_location=torch.device('cpu')))
    model.eval()
    return model

model = load_model()

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

uploaded_file = st.file_uploader("Chọn một bức ảnh từ máy của bạn...", type=["jpg", "png", "jpeg"])

if uploaded_file is not None:
    image = Image.open(uploaded_file).convert('RGB')
    st.image(image, caption='Ảnh bạn vừa tải lên', use_column_width=True)

    st.write("AI đang phân tích...")

    image_tensor = transform(image).unsqueeze(0)
    with torch.no_grad():
        outputs = model(image_tensor)
        _, predicted = torch.max(outputs, 1)
        result = class_names[predicted.item()]

    st.success(f"AI dự đoán đây là: **{result.upper()}**")

Overwriting app.py


In [20]:
!curl ipv4.icanhazip.com

35.237.243.195


In [22]:
!pip install streamlit

In [24]:
!streamlit run app.py & npx --yes localtunnel --port 8501

⠙⠹

⠸⠼⠴⠦⠧⠇your url is: https://shaky-shrimps-hide.loca.lt
2026-07-20 14:24:36.024 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.237.243.195:8501

2026-07-20 14:26:10.387 The `use_column_width` parameter has been deprecated and will be removed in a future release. Please utilize the `width` parameter instead.
2026-07-20 14:26:40.178 The `use_column_width` parameter has been deprecated and will be removed in a future release. Please utilize the `width` parameter instead.
2026-07-20 14:27:13.814 The `use_column_width` parameter has been deprecated and will be removed in a future release. Please utilize the `width` parameter instead.
  Stopping...
^C


In [25]:
%%writefile requirements.txt
streamlit
torch
torchvision
Pillow

Writing requirements.txt
